# Reproducibility smoke test on the free sample

This notebook runs `../validation/benchmark_script.py`, **unmodified and identical to the script used for the published benchmark**, against the free sample shipped in `../sample/`.

**Purpose:** show that the code runs end to end against the real delivered schema.

**Not the purpose:** reproduce the published metric. **The benchmark script cannot be run on this sample, and this notebook shows why instead of pretending otherwise.** The script imports `d04_generator`, the rule engine and interest table that ship with the generator in the full delivery. It uses them to recompute each claim's priority order and each demand's interest rate, so it cannot run from the data tables alone, and re-implementing that engine here would not be the published script.

The number to cite is the one measured on the full dataset: Task A day-0 triage AUC 0.9310 (logistic regression) and 0.9332 (gradient boosting), 5-fold GroupKFold on 5,320 claims; Task B settlement outcome AUC 0.4888 (near chance by design); Task C R2 -0.0148.


In [1]:
import sys, subprocess, shutil, tempfile, pathlib
import pandas as pd
HERE = pathlib.Path.cwd()
SAMPLE = (HERE / ".." / "sample").resolve()
SCRIPT = (HERE / ".." / "validation" / "benchmark_script.py").resolve()
work = pathlib.Path(tempfile.mkdtemp(prefix="smoke_"))   # all outputs go here, nothing is written into the repository
print("sample dir :", SAMPLE.name)
print("script     :", SCRIPT.name, "(unmodified copy of the published benchmark script)")
def clean(text):
    return text.replace(str(work), "<tmp>").replace(str(SAMPLE), "<sample>")


sample dir : sample
script     : benchmark_script.py (unmodified copy of the published benchmark script)


In [2]:
import importlib, re
try:
    importlib.import_module("d04_generator")
    print("d04_generator import: OK")
except ModuleNotFoundError as e:
    print("d04_generator import: FAILED ->", e)
print("script imports d04_generator:", "import d04_generator" in SCRIPT.read_text())
print("sample files:", sorted(p.name for p in SAMPLE.iterdir()))

d04_generator import: FAILED -> No module named 'd04_generator'
script imports d04_generator: True
sample files: ['d04_claim.csv', 'd04_claim.parquet', 'd04_coverage_period.csv', 'd04_coverage_period.parquet', 'd04_coverage_state.csv', 'd04_coverage_state.parquet', 'd04_demand.csv', 'd04_demand.parquet', 'd04_mistaken_payment.csv', 'd04_mistaken_payment.parquet', 'd04_recovery_finding.csv', 'd04_recovery_finding.parquet', 'd04_sample_passport_EN.pdf', 'd04_settlement.csv', 'd04_settlement.parquet', 'manifest.json']


## Reading the output

The import fails, as expected: the rule engine is part of the full delivery. The cells below verify what can be verified from the sample alone: file integrity against the shipped manifest, row counts, and the scenario mix.

In [3]:
import hashlib, json
manifest = json.loads((SAMPLE / "manifest.json").read_text())
rows = []
for table, info in manifest["tables"].items():
    path = SAMPLE / info["file"]
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    df = pd.read_parquet(path)
    rows.append(dict(table=table, rows_manifest=info["rows"], rows_file=len(df),
                     sha256_ok=(digest == info["sha256"])))
pd.DataFrame(rows)

,table,rows_manifest,rows_file,sha256_ok
0,claim,300,300,True
1,recovery_finding,300,300,True
2,coverage_state,404,404,True
3,mistaken_payment,104,104,True
4,demand,104,104,True
5,settlement,104,104,True
6,coverage_period,595,595,True


## Scenario mix and answer key

The `recovery_finding` answer-key table is included in the sample. Counting its scenarios shows how the 300 sampled claims are spread across the error scenarios.

In [4]:
finding = pd.read_parquet(SAMPLE / "d04_recovery_finding.parquet")
print(finding.shape)
col = [c for c in finding.columns if "scenario" in c.lower()]
print(col)
(finding[col[0]].value_counts().rename("claims").to_frame().assign(share=lambda t: (t.claims / t.claims.sum()).round(3))
 if col else finding.head())

(300, 14)
['scenario_id']


,claims,share
scenario_id,,
correct_cob,196,0.653
processing_error_known_coverage,60,0.200
retroactive_termination,22,0.073
birthday_rule_misapplied,10,0.033
late_discovery_employer,9,0.030
late_discovery_nghp,3,0.010
